<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Knowledge_Distillation_Summarize_T5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)


import torch.nn.functional as F


device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
dataset = load_dataset("LocalDoc/summarization_azerbaijan")

full_train = dataset["train"]

small_data = full_train.select(range(min(5000, len(full_train))))

small_data

In [ ]:
def clean_example(example):
    text = example["text"]
    summary = example["summary"]

    if text is None or summary is None:
        return False

    if len(text.strip()) < 50:
        return False

    if len(summary.strip()) < 5:
        return False

    return True


small_data = small_data.filter(clean_example)
small_data

In [ ]:
teacher_name = "nijatzeynalov/mT5-based-azerbaijani-summarize"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)
teacher_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_name).to(device)

teacher_model.eval()

In [ ]:
def teacher_summarize(text):
    inputs = teacher_tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = teacher_model.generate(
            **inputs,
            max_length=128,
            min_length=15,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return teacher_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [ ]:
def add_teacher_summary_batch(batch):
    texts = batch["text"]

    inputs = teacher_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        summary_ids = teacher_model.generate(
            **inputs,
            max_length=80,
            min_length=10,
            num_beams=2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    summaries = teacher_tokenizer.batch_decode(
        summary_ids,
        skip_special_tokens=True
    )

    batch["teacher_summary"] = summaries
    return batch

In [ ]:
distill_data = small_data.map(
    add_teacher_summary_batch,
    batched=True,
    batch_size=32
)

In [ ]:
# small_data = full_train.select(range(1000))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

student_name = "google/mt5-small"

student_tokenizer = AutoTokenizer.from_pretrained(student_name)

student_model = AutoModelForSeq2SeqLM.from_pretrained(
    student_name
)

student_model = student_model.to(device)
student_model.config.use_cache = False

print("Student dtype:", next(student_model.parameters()).dtype)
print("Student device:", next(student_model.parameters()).device)

In [ ]:
dataset_split = distill_data.train_test_split(
    test_size=0.05,
    seed=42
)

In [ ]:
max_input_length = 512
max_target_length = 128

prefix = "summarize: "


def preprocess_function(examples):
    inputs = [
        prefix + text
        for text in examples["text"]
    ]

    targets = examples["teacher_summary"]

    model_inputs = student_tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = student_tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_dataset = dataset_split.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_split["train"].column_names
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=student_tokenizer,
    model=student_model
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./distilled-mt5-az-summary",

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    num_train_epochs=3,
    weight_decay=0.01,

    logging_steps=50,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=False,

    fp16=False,
    bf16=False,

    max_grad_norm=1.0,

    report_to="none"
)

In [ ]:
class DistillationSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(self, teacher_model, temperature=2.0, alpha=0.7, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.teacher_model = teacher_model
        self.teacher_model.eval()

        for p in self.teacher_model.parameters():
            p.requires_grad = False

        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]

        student_outputs = model(**inputs)
        student_logits = student_outputs.logits
        ce_loss = student_outputs.loss

        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)
            teacher_logits = teacher_outputs.logits

        T = self.temperature

        student_log_probs = F.log_softmax(student_logits / T, dim=-1)

        teacher_probs = F.softmax(teacher_logits / T, dim=-1)

        kl_loss = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="none"
        )

        kl_loss = kl_loss.sum(dim=-1)

        mask = labels != -100
        kl_loss = (kl_loss * mask).sum() / mask.sum()

        kl_loss = kl_loss * (T ** 2)

        loss = self.alpha * ce_loss + (1 - self.alpha) * kl_loss

        return (loss, student_outputs) if return_outputs else loss

In [ ]:
trainer = DistillationSeq2SeqTrainer(
    teacher_model=teacher_model,
    temperature=2.0,
    alpha=0.7,

    model=student_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    processing_class=student_tokenizer
)

In [ ]:
trainer.train()

In [ ]:
def summarize_text(text):
    student_model.eval()

    input_text = prefix + text

    inputs = student_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = student_model.generate(
            **inputs,
            max_length=128,
            min_length=15,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return student_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [ ]:
my_text = """
Son illərdə süni intellekt texnologiyaları səhiyyə sahəsində daha geniş tətbiq olunmağa başlayıb.
Xəstəxanalarda tibbi görüntülərin analizi, xəstəliklərin erkən aşkarlanması və pasiyent məlumatlarının
emalı üçün müxtəlif AI modellərindən istifadə edilir. Bu texnologiyalar həkimlərə daha sürətli qərar
verməyə kömək etsə də, mütəxəssislər insan nəzarətinin vacib olduğunu bildirirlər. Çünki səhiyyədə
verilən qərarlar birbaşa insan həyatı ilə bağlıdır və modelin səhvləri ciddi nəticələrə səbəb ola bilər.
Buna görə də süni intellekt həkimi əvəz edən sistem kimi deyil, həkimə kömək edən alət kimi
qiymətləndirilməlidir.
"""

print(summarize_text(my_text))

In [ ]:
# trainer.save_model("./distilled-mt5-az-summary-final")
# student_tokenizer.save_pretrained("./distilled-mt5-az-summary-final")

In [ ]:
for i in range(5):
    print("TEXT:", small_data[i]["text"][:500])
    print("TEACHER:", teacher_summarize(small_data[i]["text"]))
    print("-" * 80)